# Results walkthrough

Reads saved artifacts only and calls package code (`optiver_tc`); it does not re-implement the pipeline.
Run from the repository root after `optiver-tc dev / freeze / final / backtest`.

In [ ]:
import os, sys, json
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, 'src')
import optiver_tc  # imports lightgbm first
import pandas as pd
from optiver_tc.config import load_config, read_json
from optiver_tc.reporting import experiment_table
cfg = load_config('configs/default.toml')

## Development experiments (folds 1-3)

In [ ]:
t = experiment_table(cfg)
t[['experiment','fold1','fold2','fold3','mean_mae','impr_vs_zero_pct','spearman_ic']].round(4)

## Frozen choices (written before the holdout was scored)

In [ ]:
fz = read_json(cfg.artifacts_dir / 'frozen_choices.json')
{k: fz[k] for k in ['frozen_at','families','lgbm_num_boost_round','ridge']} | {'w_lgb': fz['blend']['w_lgb'], 'postprocess': {k: fz['postprocess'][k] for k in ('demean','clip')}, 'rule': fz['backtest']['selected']}

## Final holdout (dates 421-480)

In [ ]:
h = read_json(cfg.artifacts_dir / 'holdout_metrics.json')
pd.DataFrame({k: h[k] for k in ['final_model','lgbm_only_raw'] + (['ridge_only'] if 'ridge_only' in h else [])}).T[['mae','rmse','dir_acc','spearman_ic_mean']]

In [ ]:
from optiver_tc import metrics as M
ho = pd.read_parquet(cfg.artifacts_dir / 'predictions' / 'holdout_predictions.parquet')
print('MAE recomputed from saved predictions:', M.mae(ho['target'], ho['pred']))
M.mae_by(ho, 'seconds_in_bucket').head(10)

## Target-based research-proxy backtest (not a live strategy)

In [ ]:
bt = read_json(cfg.artifacts_dir / 'backtest_results.json')
pd.DataFrame(bt['results']['holdout']['target_proxy']['by_cost'])[['cost_bps_per_side','turnover','mean_daily_bps','sharpe','max_drawdown_bps']]

In [ ]:
from IPython.display import Image, display
for f in ['fold_mae','feature_importance','pred_diagnostics_holdout','backtest']:
    display(Image(filename=f'reports/figures/full/{f}.png'))